# Validate Merged App Entries and Claims

- No duplicated `id` values
- Every claim points to a valid `id`
- Every claim points to its parent `id`
- No orphaned apparatus entries or claims
- Every claim should have a value in `supporting_evidence`
- Every `id` in `supporting_evidence` should resolve to an entry in the edition profile.
- The `label` and `id` in `supporting_evidence` should point to the same thing in the edition profile.

In [1]:
import json
from pathlib import Path
from collections import defaultdict
from uuid import UUID

In [2]:
def validate_uuid_urn(value):
    if not isinstance(value, str):
        raise ValueError(f"Invalid UUID URN: {value}")

    prefix = "urn:uuid:"

    if not value.startswith(prefix):
        raise ValueError(f"Invalid UUID URN: {value}")

    try:
        parsed = UUID(value[len(prefix) :])
    except ValueError as exc:
        raise ValueError(f"Malformed UUID URN: {value}") from exc

    if value != f"{prefix}{parsed}":
        raise ValueError(f"Noncanonical UUID URN: {value}")

In [3]:
with open(
    "../converted-json/merged-apparatus-entries-and-claims-updated-nested-corrected.json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

## Check for the expected structure

In [4]:
for index, entry in enumerate(data):
    if entry.get("type") != "apparatus_entry":
        raise ValueError(f"Unexpected object at index {index}: " f"{entry.get('type')}")

    if not isinstance(entry.get("claims"), list):
        raise ValueError(
            f"Entry {entry.get('id')} has a missing " "or invalid claims array"
        )

    for claim in entry["claims"]:
        if not isinstance(claim, dict):
            raise ValueError(f"Non-object claim in entry {entry['id']}")

        if claim.get("type") != "apparatus_claim":
            raise ValueError(
                f"Unexpected claim type in entry {entry['id']}: " f"{claim.get('type')}"
            )

## Check for duplicated "id" values for entries

In [5]:
entry_ids = [item["id"] for item in data]
for entry_id in entry_ids:
    validate_uuid_urn(entry_id)
if len(entry_ids) != len(set(entry_ids)):
    duplicates = set([id for id in entry_ids if entry_ids.count(id) > 1])
    raise ValueError(f"Duplicate 'id' values found in merged data: {duplicates}")

## Check for duplicated "id" values for claims

In [6]:
claim_ids = [] 
for entry in data:
    if entry.get("type") == "apparatus_entry":
        for claim in entry.get("claims", []):
            claim_ids.append(claim["id"])
            validate_uuid_urn(claim["id"])
if len(claim_ids) != len(set(claim_ids)):
    duplicates = set([id for id in claim_ids if claim_ids.count(id) > 1])
    raise ValueError(f"Duplicate 'id' values found in merged data: {duplicates}")

## Every claim should point to a corresponding apparatus entry

In [7]:
claims = [claim for entry in data if entry.get("type") == "apparatus_entry" for claim in entry.get("claims", [])]

In [8]:
entry_ids = {
    entry["id"] for entry in data if entry.get("type") == "apparatus_entry"
}
claim_targets = {
    claim["apparatus_entry_id"] for claim in claims
}
for claim_target in claim_targets:
    if claim_target in entry_ids:
        print(f"Claim points to existing apparatus entry: {claim_target}")
    elif claim_target not in entry_ids:
        raise ValueError(f"Claim points to non-existent apparatus entry: {claim_target}")

Claim points to existing apparatus entry: urn:uuid:b8b0a1da-e37c-4b6b-9e23-4144bff9ce39
Claim points to existing apparatus entry: urn:uuid:8521c41b-2ed8-41b7-a338-857bd09b40fa
Claim points to existing apparatus entry: urn:uuid:42ab9626-d180-4f38-b66e-cc0795a7f54b
Claim points to existing apparatus entry: urn:uuid:20714735-1595-4b9d-865a-fb071e3da43a
Claim points to existing apparatus entry: urn:uuid:64de5bba-d578-493d-b9c3-0c52b33dd155
Claim points to existing apparatus entry: urn:uuid:c8f6d0a9-7c57-4a5e-9def-3ca62cf2e188
Claim points to existing apparatus entry: urn:uuid:b61264a3-94b9-44ea-9a2a-5a84cc652da4
Claim points to existing apparatus entry: urn:uuid:3e72d73b-2f5f-469b-b053-1ffa42edcb6c
Claim points to existing apparatus entry: urn:uuid:390d0c28-f0c2-47a7-a75d-f0fd1b9d5601
Claim points to existing apparatus entry: urn:uuid:415f78c3-5634-4117-b2ce-bc1fab969157
Claim points to existing apparatus entry: urn:uuid:d2d78ce8-ae8b-4e6f-be2b-e8b21ea4fce2
Claim points to existing apparat

## Every claim points to its parent `id`

In [9]:
for entry in data:
    for claim in entry.get("claims", []):
        if claim["apparatus_entry_id"] != entry["id"]:
            raise ValueError(
                f"Claim {claim['id']} points to apparatus entry {claim['apparatus_entry_id']}, but is nested under apparatus entry {entry['id']}"
            )

## No orphaned apparatus entries or claims

In [10]:
for entry in data:
    if entry["claims"] == []:
        print(f"Apparatus entry {entry['id']} has no claims.")

Apparatus entry urn:uuid:bf9a7a3f-e6b6-40f9-b109-516575b716da has no claims.
Apparatus entry urn:uuid:769ca300-38ab-494c-91db-c5bc0f819da7 has no claims.
Apparatus entry urn:uuid:7ae6a99d-1ac1-499e-847f-283b984dd903 has no claims.


These are unusual entries. I'll handle them manually.

In [11]:
# Make sure that all claims have a corresponding apparatus entry
orphaned_claims = [
    claim for claim in data if claim.get("type") == "apparatus_claim" and claim["apparatus_entry_id"] not in entry_ids
]
if orphaned_claims:
    raise ValueError(f"Orphaned claims found: {orphaned_claims}")

In [12]:
# Every claim should have a value in the "supporting_evidence" field
no_evidence_claims = [
    claim for claim in claims if not claim.get("supporting_evidence")
]
if no_evidence_claims:
    raise ValueError(f"Claims without supporting evidence found: {no_evidence_claims}")

## Every `id` in `supporting_evidence` should resolve to an entry in the edition profile.

In [13]:
# Open the edition profile
with open("../converted-json/phi0830.phi001.new_edition_profile2.json", "r", encoding="utf-8") as f:
    edition_profile = json.load(f)

In [14]:
# Get the ids of all supporting evidence
supporting_evidence_ids = []
for claim in claims:
    supporting_evidence = claim.get("supporting_evidence", [])
    for evidence in supporting_evidence:
        if "id" in evidence:
            supporting_evidence_ids.append(evidence["id"])
supporting_evidence_id_set = set(supporting_evidence_ids)

# Get the uuids of all the items in the edition profile
edition_profile_uuids = []
items = ["sigla", "secondary_sources", "people"]
for item in items:
    for sub_item in edition_profile[item]:
        if "uuid" in sub_item:
            edition_profile_uuids.append(sub_item["uuid"])
edition_profile_uuid_set = set(edition_profile_uuids)

for evidence_id in supporting_evidence_id_set:
    if evidence_id not in edition_profile_uuid_set:
        raise ValueError(f"Supporting evidence id {evidence_id} not found in edition profile")

I should see if there are any uncited sources.

In [15]:
uncited_sources = []
for source in edition_profile_uuid_set:
    if source not in supporting_evidence_id_set:
        uncited_sources.append(source)

In [16]:
len(uncited_sources)

74

Well, that's not good. I should see which sources those are.

In [17]:
uncited_items = []
items = ["sigla", "secondary_sources", "people"]
for item in items:
    for sub_item in edition_profile[item]:
        if sub_item.get("uuid") in uncited_sources:
            uncited_items.append(sub_item)
            print(sub_item["label"] if "label" in sub_item else sub_item.get("name", "No name or abbreviation"))

ζ
cod. Vindob.
Exc. Par. Prior
Exc. Par. Alter
Exc. Par.
Exc. Bon.
d1
c
Nordh.
i1
i2
l1
l2
Aurel.
Giarratano
Baehr. 1872
Barth 1650
Bartholinus
Bergk
Brantsma
Buecheler 1860
Buecheler 1871
Buecheler 1907
Burm. 1759
Kannegieter
Carrio
Chytil
de Rooy
Erasmus
Fabricius
Forbiger
Fritzsche
Gebauer
Gebhard
Gronov. 1637
Lucas
Haupt 1854
Heins. in Her.
Heins. ad Claud.
G. Hermann
Hoeufft
Housman 1902
Housman 1903
Iustus
Magnus
Modius
Salmasius 1620
Salmasius 1656
Sarpe
Spanheim
Swartius
Tross
Voss
Wakefield
Argol.
Ascensius
Barthius
Brodaeus
Bursian
D’Orville
Casaub.
Fruterius
Gudius
Housman
Lachmann
Lipsius
Oudendorp
Ramorino
Scaliger
C. Schenkl
Scriver.
Schraeder
Tortell.
Wolf


I'm comfortable with saying that Giarratano did not cite those sources in the apparatus to the first poem.

## The `label` and `id` in `supporting_evidence` should point to the same thing in the edition profile.

This is a little bit more complex.

In [18]:
concordance = {}

for item in items:
    for sub_item in edition_profile[item]:
        sub_item_id = sub_item.get("uuid")
        sub_item_label = sub_item.get("label")

        if not sub_item_id or not sub_item_label:
            raise ValueError(f"Missing uuid or label in {item}: {sub_item}")

        if sub_item_id in concordance:
            raise ValueError(f"Duplicate UUID in edition profile: {sub_item_id}")

        concordance[sub_item_id] = sub_item_label

In [19]:
for claim in claims:
    evidence = claim.get("supporting_evidence", [])
    if not isinstance(evidence, list):
        raise ValueError(f"Invalid supporting_evidence in {claim['id']}")

    if not evidence:
        print(f"Review: claim {claim['id']} has no " "supporting evidence")
    for evidence in claim.get("supporting_evidence", []):
        evidence_id = evidence.get("id")
        evidence_label = evidence.get("label")
        if "group" in evidence:
            continue  # Skip validation for grouped evidence
        if evidence_id in concordance:
            if evidence_label != concordance[evidence_id]:
                raise ValueError(f"Supporting evidence label mismatch for id {evidence_id}: {evidence_label} vs {concordance[evidence_id]}")
        elif evidence_id is None:
            raise ValueError(f"Supporting evidence id {evidence_id} is None")
        else:
            raise ValueError(f"Supporting evidence id {evidence_id} not found in concordance")

In [20]:
# Check groups, which are not checked in the previous loop
for claim in claims:
    for evidence in claim["supporting_evidence"]:
        if "group" not in evidence:
            continue

        group = evidence["group"]

        # Require an explicit boundary field
        if "boundary" not in group:
            raise ValueError(f"Missing boundary in claim {claim['id']}")

        boundary = group["boundary"]

        # A null boundary is valid
        if boundary is None:
            continue

        # Otherwise, boundary must be a dictionary
        if not isinstance(boundary, dict):
            raise ValueError(f"Invalid group boundary in claim {claim['id']}")

        boundary_id = boundary.get("id")
        boundary_label = boundary.get("label")

        if boundary_id is None:
            if boundary_label is not None:
                raise ValueError(f"Boundary label without ID in {claim['id']}")
            continue

        if boundary_id not in concordance:
            raise ValueError(f"Unknown boundary ID: {boundary_id}")

        if boundary_label != concordance[boundary_id]:
            raise ValueError(f"Boundary label mismatch: {boundary_label}")

## Check the order of reporting ranks

In [21]:
for entry in data:
    ranks = []

    for claim in entry["claims"]:
        reading_type = claim.get("reading_type")
        rank = claim.get("reporting_rank")

        if reading_type not in {"lemma", "variant"}:
            raise ValueError(f"Invalid reading type in {claim['id']}")

        if reading_type == "lemma":
            if rank is not None:
                raise ValueError(f"Lemma has reporting rank: {claim['id']}")

        elif reading_type == "variant":
            if type(rank) is not int or rank < 1:
                raise ValueError(f"Invalid variant rank: {claim['id']}")
            ranks.append(rank)

    if len(ranks) != len(set(ranks)):
        raise ValueError(f"Duplicate reporting ranks in entry {entry['id']}")

## Validate the form of `text_locus`

It should be a list, not a string.

In [22]:
for entry in data:
    if not isinstance(entry["text_locus"], list):
        raise ValueError(f"Invalid text_locus in entry {entry['id']}")